# **Classification: Stacking Classifier**

## **Justification of Algorithm Selection**

Stacking involves training a meta-model to intelligently combine the predictions of several distinct base models. The strength of a stack relies entirely on the **diversity** and **robustness** of its members. We must select models that make different types of mathematical mistakes, allowing the meta-model to learn when to trust which expert. Based on our extensive prior experiments, we have curated the following "Elite Team":

* **Logistic Regression:** Our linear, mathematical champion. Highly stable and calibrated.
* **Naive Bayes:** Our probabilistic expert. It looks at the data through the lens of conditional independence.
* **Random Forest:** Our regularized bagging champion. It uses a "wisdom of the crowd" approach with deep, diverse trees.
* **AdaBoost & Gradient Boosting (GBC):** Our sequential boosting experts. They learn by correcting the errors of prior iterations using shallow stumps/trees.
* **XGBoost:** The advanced, highly optimized gradient boosting framework.

### **The Excluded Models (What We Left Out and Why)**
Strategic exclusion is just as important as inclusion. We rejected the following models to protect the integrity and performance of the stack:
* **K-Nearest Neighbors (K-NN):** Rejected due to severe overfitting ($1.0$ Train Recall) and an unacceptable prediction latency (the "lazy learner" bottleneck) that would cripple our real-time Streamlit application.
* **Decision Tree:** Redundant. The Random Forest already represents the optimal evolution of our Decision Tree structure.
* **Bagging Classifier:** Redundant and computationally heavy. We already have Random Forest for tree-based bagging, and bagging the Logistic Regression inside a stack is an unnecessary nested ensemble.
* **Support Vector Classifier (SVC):** Rejected due to strict hardware limitations. SVC scaled poorly past 50,000 rows. Including it in a Stacking Classifier (which uses internal 3-fold cross-validation) would result in unfeasible execution times spanning several days.


## **Preprocessing: The Scale Sensitivity of a Heterogeneous Stack**

Our ensemble features a distance-dependent model (Logistic Regression), a probability model (Naive Bayes), and multiple scale-invariant tree-based models. 
Because the meta-model itself is a Logistic Regression (which requires scaled inputs to assign proper weights to the base models' predictions), feature scaling is strictly mandatory for the entire pipeline. 
We will test both **Standardization (`StandardScaler`)** and **Normalization (`MinMaxScaler`)** to determine which continuous space allows the meta-model to best harmonize these diverse experts.


## **Experiment Design**

Following the strategy of reusing "Champion" parameters, we eliminate the need for base-level hyperparameter optimization. We lock in the exact parameters that yielded the best generalized Recall in our previous notebooks. 

We define a tournament of **2 focused experiments**:
* **Standardized Champion Stack**: Evaluated on data scaled via `StandardScaler`.
* **Normalized Champion Stack**: Evaluated on data scaled via `MinMaxScaler`.

We strictly log **both Train and Test metrics** to ensure the meta-model successfully aggregates the experts without triggering data leakage or cascading overfitting.

In [ ]:
import pandas as pd
import numpy as np
import time
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import recall_score, accuracy_score, f1_score

# 1. MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_Stacking")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")
categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level', 'employment_status', 'age_groups', 'weight_status', 'income_level']
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Split data (80/20) maintaining class proportion
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

def log_classification_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to explicitly monitor the Overfitting Gap"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Partition Metrics
    mlflow.log_metric("recall_train", recall_score(y_tr, y_tr_pred))
    mlflow.log_metric("accuracy_train", accuracy_score(y_tr, y_tr_pred))
    mlflow.log_metric("f1_train", f1_score(y_tr, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("recall_test", recall_score(y_te, y_te_pred))
    mlflow.log_metric("accuracy_test", accuracy_score(y_te, y_te_pred))
    mlflow.log_metric("f1_test", f1_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# DEFINING THE CHAMPION MODELS (Fixed winners from previous notebooks)
# ---------------------------------------------------------

champion_xgb = XGBClassifier(
    booster='gbtree',
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=100,
    learning_rate=0.3,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

champion_rf = RandomForestClassifier(
    n_estimators=150,
    criterion='gini',
    max_depth=10,
    min_samples_leaf=50,
    max_samples=0.7,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

champion_gbc = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=1.0,
    random_state=42
)

champion_ada = AdaBoostClassifier(
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)

champion_lr = LogisticRegression(
    solver='saga',
    penalty='l2',
    C=0.0021351067557515042,
    max_iter=2000,
    random_state=42
)

champion_nb = GaussianNB(
    var_smoothing=0.013229671831047198
)

# List of base specialists for the Stack
champions_list = [
    ('xgb', champion_xgb),
    ('rf', champion_rf),
    ('gbc', champion_gbc),
    ('ada', champion_ada),
    ('lr', champion_lr),
    ('nb', champion_nb)
]

# ---------------------------------------------------------
# TOURNAMENT: 2 RUNS (Standardization vs Normalization)
# ---------------------------------------------------------
scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"Stacking_{s_name}_Champions"):
        # Apply scaling to numerical features
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])

        # Initialize Stacking with fixed champions and default meta-model
        stack_model = StackingClassifier(
            estimators=champions_list, 
            final_estimator=LogisticRegression(),
            n_jobs=-1,
            cv=3 # Internal cross-validation to prevent leakage from base models
        )
        start_time = time.time()
        stack_model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        # MLflow logging
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("meta_model", "LogisticRegression_Default")
        mlflow.log_param("optimization", "fixed_champions_team")
        
        # Log specific champion parameters for transparency
        mlflow.log_param("team_size", len(champions_list))
        mlflow.log_param("included_models", "XGB, RF, GBC, ADA, LR, NB")
            
        log_classification_metrics(stack_model, X_train_scaled, y_train, X_test_scaled, y_test, duration)

## Winner Run Selection (Priority Elimination Framework)

### New Policy (effective immediately)
A run is only eligible to win if it does **not** show evidence of overfitting or underfitting. Before applying the Recall/F1/fit_time decision rules, we require the **Recall** and **F1** Train→Test gaps (Test − Train) to remain within ±0.5 percentage points (|gap| ≤ 0.005) to consider a run as generalizing. If a run fails this check it is disqualified regardless of metric rank.

### Selection Criteria (priority order)
1. **Generalization filter (mandatory):** Recall and F1 gaps within ±0.5 percentage points. Disqualified runs are removed from consideration.
2. **Priority 1 (70%): Highest Recall (Test)** — clinical priority: maximize detection of positive diabetes cases.
3. **Priority 2 (30%): Highest F1-Score (Test)** — used when Recall ties or differs by <0.5% among remaining candidates.
4. **Accuracy is visible but ignored** — shown for reference only; not used in selection.
5. **Tiebreaker: Lowest Fit Time** — if Recall and F1 remain tied.

### Runs Summary

| Run | Scaler | Accuracy (Train) | Accuracy (Test) | Recall (Train) | Recall (Test) | F1 (Train) | F1 (Test) | Fit Time |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| Stacking_Standardization_Champions | Standardization | 0.92179 | 0.91965 | 0.86964 | 0.86675 | 0.93028 | 0.92829 | 66.44s |
| Stacking_Normalization_Champions | Normalization | 0.92181 | 0.91965 | 0.86968 | 0.86675 | 0.93030 | 0.92829 | 63.83s |

### Generalization Check (Test − Train)
- **Stacking_Standardization_Champions:** Recall gap = 0.86675 − 0.86964 = **−0.29pp** → PASS. F1 gap = 0.92829 − 0.93028 = **−0.20pp** → PASS.
- **Stacking_Normalization_Champions:** Recall gap = 0.86675 − 0.86968 = **−0.29pp** → PASS. F1 gap = 0.92829 − 0.93030 = **−0.20pp** → PASS.

### Step-by-Step Elimination
**Step 1 — Apply the generalization filter**
- Passing runs: **both runs**.

**Step 2 — Compare Test Recall (Priority 1 — 70%)**
- Both runs have identical Test Recall: **0.86675**.
- Proceed to Priority 2.

**Step 3 — Compare Test F1 (Priority 2 — 30%)**
- Both runs have effectively identical Test F1: **0.9282877415**.
- Proceed to tiebreaker.

**Step 4 — Fit Time tiebreaker**
- `Stacking_Standardization_Champions`: **66.44s**
- `Stacking_Normalization_Champions`: **63.83s**

### Final Decision
**Winner: Stacking_Normalization_Champions**

**Justification:** Both runs satisfy the generalization filter and have the same predictive performance on the test set, so the decision is resolved by fit time. `Stacking_Normalization_Champions` is slightly faster and therefore the best operational choice.

## Winner Hyperparameters

| Parameter | Value |
|---|---|
| **Scaler** | Normalization |
| **Meta-model** | LogisticRegression(Default) |
| **Team size** | 6 |
| **Included models** | XGB, RF, GBC, ADA, LR, NB |
| **Optimization** | fixed_champions_team |

## Overfitting / Underfitting Diagnosis
- Both runs show small Train→Test gaps on Recall and F1, so there is **no disqualifying overfitting or underfitting** under the current policy.
- Because the metrics are tied, the faster run wins.

**Operational note:** This selection is based on manual analysis of the logged metrics, not on scripts.